# S4Casting: Data Preparation

This Source Code Form is subject to the terms of the Mozilla Public  License, v. 2.0. If a copy of the MPL was not distributed with this file, You can obtain one at http://mozilla.org/MPL/2.0/.
This notebook contains scripts and utilities for preparing and formatting datasets for use with the S4Casting framework. Proper data formatting is crucial for effective model training and evaluation.

## Overview

Time series datasets are typically stored in **Parquet** files, with a **time column** and a **target column**. But S4Casting requires the data to be in a specific format to ensure compatibility with its training and evaluation pipelines, for efficient sampling and processing. 

This includes:

### 1. `.np` File (NumPy Memory-Mapped Array)
- Contains a **1D NumPy array** with all historical measurement values. Each value corresponds to an active power reading.

### 2. `.parquet` File (Span Metadata)
- Stores **spans** of measurement data, each representing a time window for a specific location.

### 3. `.json` File (Dataset Descriptor)
- Connects the `.np` and `.parquet` files
- Includes metadata for locations

> **Note:**  
> When opening VS Code in SageMaker for the first time, the Python kernel may not appear.  
> Simply press **Ctrl + R** in your browser to reload the page and the kernel should be detected.



In [ ]:
import json
import os
import pprint
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add project root to Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import utilities
from s4casting.data.preparation.dataset_formatter import DatasetFormatter
from tests.utils import create_sinusoid_test_data_raw

print("✓ Imports successful!")

## Step 1: Create Raw Data

For this example, we'll generate synthetic sinusoidal data to demonstrate the preprocessing pipeline.
In practice, you would start with your own time series data in Parquet or CSV format.

In [ ]:
# Create synthetic data for demonstration
create_sinusoid_test_data_raw("data/sinusoid_data_raw")

# List the generated files
raw_data_path = "data/sinusoid_data_raw/"
files = os.listdir(raw_data_path)
print(f"Generated {len(files)} location files:")
for f in files:
    print(f"  - {f}")

## Step 2: Inspect Raw Data

Let's examine the structure of the raw data files.

In [ ]:
# Load one example file
df = pd.read_parquet("data/sinusoid_data_raw/location_a.parquet")

print("Raw data structure:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print("\nFirst few rows:")
display(df.head())

# Visualize the time series
fig, ax = plt.subplots(figsize=(12, 4))
# Plot 8 days of data (2400 samples at 5-min intervals)
ax.plot(df["timestamp"][:2400], df["measurements"][:2400], linewidth=1)
ax.set_title("Example Time Series from Location A (First 8 Days)", fontsize=14, fontweight="bold")
ax.set_xlabel("Timestamp", fontsize=12)
ax.set_ylabel("Measurements", fontsize=12)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step 3: Format the Dataset

Now we'll convert the raw data into the S4Casting format using the `DatasetFormatter`.

This process:
- Reads all parquet/csv files in the input folder
- Converts timestamps to Unix time
- Creates spans for each location
- Writes a memory-mapped numpy array
- Generates metadata files

Dataset assumptions:
- All input files have consistent schema with a `time_col` and a `target_col`.
- The time column is in UTC.
- Target column is in MW.
- Each file represents one location (filename stem is used as location name).
- Folder contains only relevant .parquet or .csv files.
- One contiguous span per file: num_values = number of rows.
- No gap detection or multiple spans per location.


In [ ]:
output_dir = "data/notebook_test_data/"

# Run the formatter
formatter = DatasetFormatter(
    folder="data/sinusoid_data_raw",
    output_prefix="external_data_wrapped",
    output_dir=output_dir,
    target_col="measurements",
    time_col="timestamp",
    sample_interval_minutes=5,
)

formatter.run()

## Step 4: Verify the Formatted Data

Let's inspect the generated files to ensure everything is correct, read [DATA SOURCES](../docs/DATA_SOURCES.md) for the details of the data sources.

In [ ]:
print("\n" + "=" * 60)
print("JSON METADATA")
print("=" * 60)

with Path(f"{output_dir}/external_data_wrapped.json").open(encoding="utf-8") as f:
    json_data = json.load(f)

print("\nDataset configuration:")
pprint.pprint(json_data, indent=2, width=80)

print("\n" + "=" * 60)
print("MEMORY-MAPPED DATA ARRAY")
print("=" * 60)

data_memmap = np.memmap(f"{output_dir}/external_data_wrapped.np", mode="r", dtype="float32")

print(f"\nArray shape: {data_memmap.shape} \n")

print("=" * 60)
print("SPANS METADATA")
print("=" * 60)

spans_df = pd.read_parquet(f"{output_dir}/external_data_wrapped_spans.parquet")
print(f"\nShape: {spans_df.shape}")
print(f"Columns: {list(spans_df.columns)}")
print("\nFirst few spans:")
display(spans_df.head())

## Summary & Next Steps

You've successfully formatted your time series data for S4Casting! 

### Next Steps
1. Use the formatted dataset for training S4Casting models in `01_train_model.ipynb`.
2. Try to format your own datasets using the `DatasetFormatter`.
